In [1]:
import requests
import pandas as pd
import time
import os
import random
import re
import unicodedata
from urllib.parse import quote # Para codificar URLs

In [ ]:
ARQUIVO_PT_API_BASE_URL = "https://arquivo.pt/textsearch"
USER_AGENT = "Interior+/1.0 (danielmsrcarvalho.com)" # Substituir com o teu email
REQUEST_DELAY_SECONDS = 0.8 # Aumentar se houver problemas de rate limiting
JITTER_SECONDS = 0.3
REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 5
RETRY_BACKOFF_BASE = 1.8
RETRY_BACKOFF_MAX = 90
RETRY_JITTER_SECONDS = 0.5
WORKER_TAG = "C" # Ex: "A" ou "1996-2005"
OUTPUT_DIR = "dados_arquivopt"

In [3]:
def polite_sleep():
    delay = REQUEST_DELAY_SECONDS + random.uniform(0, JITTER_SECONDS)
    time.sleep(delay)

In [4]:
# Lista expandida de keywords
event_keywords = {
    "eventos": [
        "programa de apoio ao interior",
    ]
}

# Intervalo de anos para pesquisa
YEAR_START = 1996 # Início da indexação do Arquivo.pt relevante
YEAR_END = 2026 # Ano atual

In [ ]:
def build_api_url(query: str, start_date: str, end_date: str, page: int = 1, limit: int = 1000) -> str:
    """
    Constroi a URL para a API de pesquisa publica do Arquivo.pt (textsearch).
    Mais info: https://github.com/arquivo/cdx-api-crawler
    """

    search_url = (
        f"{ARQUIVO_PT_API_BASE_URL}?q={quote(query)}"
        f"&from={start_date}&to={end_date}"
        f"&maxItems={limit}"
        f"&page={page}"
        f"&output=json"
    )
    return search_url

In [6]:
def fetch_data_from_api(query: str, year: int) -> list:
    """
    Faz pedidos a API do Arquivo.pt para uma dada query e ano,
    gerindo a paginacao e o atraso entre pedidos.
    """
    all_results = []
    page_count = 0
    item_count = 0

    # Formato de data para a API (YYYYMMDDhhmmss)
    start_date_str = f"{year}0101000000"
    end_date_str = f"{year}1231235959"

    print(f"A procurar por '{query}' para o ano {year}...")

    next_url = build_api_url(query, start_date_str, end_date_str, page=1)
    headers = {"User-Agent": USER_AGENT, "Accept": "application/json"}

    while next_url:
        try:
            page_count += 1
            response = None
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    response = requests.get(next_url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
                    if response.status_code in (429, 500, 502, 503, 504):
                        if attempt == MAX_RETRIES:
                            response.raise_for_status()
                        retry_after = response.headers.get("Retry-After")
                        if retry_after and retry_after.isdigit():
                            backoff = float(retry_after)
                        else:
                            backoff = min(RETRY_BACKOFF_MAX, RETRY_BACKOFF_BASE ** attempt)
                            backoff += random.uniform(0, RETRY_JITTER_SECONDS)
                        print(f"  HTTP {response.status_code}. A tentar novamente em {backoff:.1f}s (tentativa {attempt}/{MAX_RETRIES})...")
                        time.sleep(backoff)
                        continue
                    break
                except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
                    if attempt == MAX_RETRIES:
                        raise
                    backoff = min(RETRY_BACKOFF_MAX, RETRY_BACKOFF_BASE ** attempt)
                    backoff += random.uniform(0, RETRY_JITTER_SECONDS)
                    print(f"  Timeout/ligacao. A tentar novamente em {backoff:.1f}s (tentativa {attempt}/{MAX_RETRIES})...")
                    time.sleep(backoff)
            response.raise_for_status() # Lanca excecao para erros HTTP (4xx ou 5xx)
            content_type = response.headers.get("Content-Type", "")
            if "json" not in content_type.lower():
                print(f"Resposta nao-JSON para '{query}' (ano {year}).")
                print(response.text[:300])
                break

            data = response.json()
            items = data.get("response_items") or data.get("responseItems") or []

            if items:
                item_count += len(items)
                for item in items:
                    original_url = (
                        item.get("originalURL")
                        or item.get("originalUrl")
                        or item.get("original_url")
                    )
                    archive_url = (
                        item.get("linkToArchive")
                        or item.get("link_to_archive")
                        or item.get("archive_url")
                    )
                    archive_date = item.get("date") or item.get("timestamp")
                    result = {
                        "query": query,
                        "original_url": original_url,
                        "archive_url": archive_url,
                        "title": item.get("title"),
                        "snippet": item.get("snippet"),
                        "archive_date": archive_date, # Formato YYYYMMDDhhmmss
                        "year": year
                    }
                    all_results.append(result)
            
            # A API de pesquisa de texto indica a proxima pagina em next_page
            next_page = data.get("next_page")
            if next_page:
                if isinstance(next_page, str) and next_page.startswith("http"):
                    next_url = next_page
                else:
                    next_url = build_api_url(query, start_date_str, end_date_str, page=next_page)
                polite_sleep()
            else:
                next_url = None

        except requests.exceptions.RequestException as e:
            print(f"Erro ao fazer pedido a API para '{query}' (ano {year}): {e}")
            break
        except ValueError:
            print(f"Erro ao decodificar JSON para '{query}' (ano {year}).")
            print(response.text[:300])
            break

    if not all_results:
        print(f"  Nenhum resultado encontrado para '{query}' no ano {year}.")
    print(f"  Total de {len(all_results)} resultados para '{query}' no ano {year}.")
    print(f"  Paginas obtidas: {page_count}; itens na resposta: {item_count}.")
    return all_results

In [ ]:
# Teste de sanidade para garantir que a função de busca está a funcionar antes de correr o processo completo
test_query = "agricultura"
test_year = YEAR_START
test_results = fetch_data_from_api(test_query, test_year)
print(f"Sanity test: {len(test_results)} resultados para '{test_query}' em {test_year}.")

In [7]:
def slugify_keyword(keyword: str) -> str:
    normalized = unicodedata.normalize("NFKD", keyword)
    ascii_text = normalized.encode("ascii", "ignore").decode("ascii")
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", ascii_text).strip("_").lower()
    return cleaned or "keyword"


def run_year(year: int) -> list:
    output_files = []

    for event_type, keywords in event_keywords.items():
        for kw in keywords:
            results = fetch_data_from_api(kw, year)
            if results:
                df = pd.DataFrame(results)

                # Remover duplicados, pois a mesma pagina pode ser encontrada por diferentes queries
                # ou em diferentes paginas da API para a mesma query/ano.
                # Considera archive_url e archive_date como identificadores unicos
                df.drop_duplicates(subset=["archive_url", "archive_date"], inplace=True)

                tag = f"_{WORKER_TAG}" if WORKER_TAG else ""
                keyword_slug = slugify_keyword(kw)
                output_filename = os.path.join(OUTPUT_DIR, f"{keyword_slug}_{year}{tag}.csv")
                df.to_csv(output_filename, index=False, encoding="utf-8-sig") # utf-8-sig para bom suporte a caracteres especiais
                print(f"\nDados recolhidos guardados em: {output_filename}")
                print(f"Total de registos unicos: {len(df)}")
                output_files.append(output_filename)

            # Pequeno atraso entre diferentes queries também
            polite_sleep()

    if not output_files:
        print("Nao foram encontrados dados para as queries e ano especificados.")
    return output_files

In [79]:
# Ano 1996
run_year(1996)

A procurar por 'programa de apoio ao interior' para o ano 1996...
  Total de 13 resultados para 'programa de apoio ao interior' no ano 1996.
  Paginas obtidas: 1; itens na resposta: 13.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_1996_C.csv
Total de registos unicos: 13


['dados_arquivopt\\programa_de_apoio_ao_interior_1996_C.csv']

In [80]:
# Ano 1997
run_year(1997)

A procurar por 'programa de apoio ao interior' para o ano 1997...
  Total de 144 resultados para 'programa de apoio ao interior' no ano 1997.
  Paginas obtidas: 1; itens na resposta: 144.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_1997_C.csv
Total de registos unicos: 144


['dados_arquivopt\\programa_de_apoio_ao_interior_1997_C.csv']

In [81]:
# Ano 1998
run_year(1998)

A procurar por 'programa de apoio ao interior' para o ano 1998...
  Total de 309 resultados para 'programa de apoio ao interior' no ano 1998.
  Paginas obtidas: 1; itens na resposta: 309.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_1998_C.csv
Total de registos unicos: 309


['dados_arquivopt\\programa_de_apoio_ao_interior_1998_C.csv']

In [82]:
# Ano 1999
run_year(1999)

A procurar por 'programa de apoio ao interior' para o ano 1999...
  Total de 338 resultados para 'programa de apoio ao interior' no ano 1999.
  Paginas obtidas: 1; itens na resposta: 338.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_1999_C.csv
Total de registos unicos: 338


['dados_arquivopt\\programa_de_apoio_ao_interior_1999_C.csv']

In [83]:
# Ano 2000
run_year(2000)

A procurar por 'programa de apoio ao interior' para o ano 2000...
  Total de 500 resultados para 'programa de apoio ao interior' no ano 2000.
  Paginas obtidas: 2; itens na resposta: 500.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2000_C.csv
Total de registos unicos: 500


['dados_arquivopt\\programa_de_apoio_ao_interior_2000_C.csv']

In [84]:
# Ano 2001
run_year(2001)

A procurar por 'programa de apoio ao interior' para o ano 2001...
  Total de 571 resultados para 'programa de apoio ao interior' no ano 2001.
  Paginas obtidas: 2; itens na resposta: 571.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2001_C.csv
Total de registos unicos: 571


['dados_arquivopt\\programa_de_apoio_ao_interior_2001_C.csv']

In [85]:
# Ano 2002
run_year(2002)

A procurar por 'programa de apoio ao interior' para o ano 2002...
  Total de 578 resultados para 'programa de apoio ao interior' no ano 2002.
  Paginas obtidas: 2; itens na resposta: 578.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2002_C.csv
Total de registos unicos: 578


['dados_arquivopt\\programa_de_apoio_ao_interior_2002_C.csv']

In [86]:
# Ano 2003
run_year(2003)

A procurar por 'programa de apoio ao interior' para o ano 2003...
  Total de 724 resultados para 'programa de apoio ao interior' no ano 2003.
  Paginas obtidas: 2; itens na resposta: 724.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2003_C.csv
Total de registos unicos: 724


['dados_arquivopt\\programa_de_apoio_ao_interior_2003_C.csv']

In [87]:
# Ano 2004
run_year(2004)

A procurar por 'programa de apoio ao interior' para o ano 2004...
  Total de 808 resultados para 'programa de apoio ao interior' no ano 2004.
  Paginas obtidas: 2; itens na resposta: 808.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2004_C.csv
Total de registos unicos: 808


['dados_arquivopt\\programa_de_apoio_ao_interior_2004_C.csv']

In [88]:
# Ano 2005
run_year(2005)

A procurar por 'programa de apoio ao interior' para o ano 2005...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2005.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2005_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2005_C.csv']

In [89]:
# Ano 2006
run_year(2006)

A procurar por 'programa de apoio ao interior' para o ano 2006...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2006.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2006_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2006_C.csv']

In [90]:
# Ano 2007
run_year(2007)

A procurar por 'programa de apoio ao interior' para o ano 2007...
  Total de 500 resultados para 'programa de apoio ao interior' no ano 2007.
  Paginas obtidas: 2; itens na resposta: 500.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2007_C.csv
Total de registos unicos: 500


['dados_arquivopt\\programa_de_apoio_ao_interior_2007_C.csv']

In [91]:
# Ano 2008
run_year(2008)

A procurar por 'programa de apoio ao interior' para o ano 2008...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2008.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2008_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2008_C.csv']

In [92]:
# Ano 2009
run_year(2009)

A procurar por 'programa de apoio ao interior' para o ano 2009...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2009.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2009_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2009_C.csv']

In [8]:
# Ano 2010
run_year(2010)

A procurar por 'programa de apoio ao interior' para o ano 2010...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2010.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2010_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2010_C.csv']

In [9]:
# Ano 2011
run_year(2011)

A procurar por 'programa de apoio ao interior' para o ano 2011...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2011.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2011_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2011_C.csv']

In [10]:
# Ano 2012
run_year(2012)

A procurar por 'programa de apoio ao interior' para o ano 2012...
  Total de 1000 resultados para 'programa de apoio ao interior' no ano 2012.
  Paginas obtidas: 3; itens na resposta: 1000.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2012_C.csv
Total de registos unicos: 1000


['dados_arquivopt\\programa_de_apoio_ao_interior_2012_C.csv']

In [11]:
# Ano 2013
run_year(2013)

A procurar por 'programa de apoio ao interior' para o ano 2013...
  Total de 1000 resultados para 'programa de apoio ao interior' no ano 2013.
  Paginas obtidas: 3; itens na resposta: 1000.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2013_C.csv
Total de registos unicos: 1000


['dados_arquivopt\\programa_de_apoio_ao_interior_2013_C.csv']

In [12]:
# Ano 2014
run_year(2014)

A procurar por 'programa de apoio ao interior' para o ano 2014...
  Total de 1000 resultados para 'programa de apoio ao interior' no ano 2014.
  Paginas obtidas: 3; itens na resposta: 1000.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2014_C.csv
Total de registos unicos: 1000


['dados_arquivopt\\programa_de_apoio_ao_interior_2014_C.csv']

In [13]:
# Ano 2015
run_year(2015)

A procurar por 'programa de apoio ao interior' para o ano 2015...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2015.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2015_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2015_C.csv']

In [14]:
# Ano 2016
run_year(2016)

A procurar por 'programa de apoio ao interior' para o ano 2016...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2016.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2016_C.csv
Total de registos unicos: 993


['dados_arquivopt\\programa_de_apoio_ao_interior_2016_C.csv']

In [15]:
# Ano 2017
run_year(2017)

A procurar por 'programa de apoio ao interior' para o ano 2017...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2017.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2017_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2017_C.csv']

In [16]:
# Ano 2018
run_year(2018)

A procurar por 'programa de apoio ao interior' para o ano 2018...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2018.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2018_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2018_C.csv']

In [17]:
# Ano 2019
run_year(2019)

A procurar por 'programa de apoio ao interior' para o ano 2019...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2019.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2019_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2019_C.csv']

In [18]:
# Ano 2020
run_year(2020)

A procurar por 'programa de apoio ao interior' para o ano 2020...
  Total de 1001 resultados para 'programa de apoio ao interior' no ano 2020.
  Paginas obtidas: 3; itens na resposta: 1001.

Dados recolhidos guardados em: dados_arquivopt\programa_de_apoio_ao_interior_2020_C.csv
Total de registos unicos: 1001


['dados_arquivopt\\programa_de_apoio_ao_interior_2020_C.csv']